# ORB Strategy — Post-Run Analysis

**Usage:** Run cells top-to-bottom. Cell 1 lists all available files in ObjectStore. Set `RUN_KEY` in Cell 2:
- `'latest'` — loads the most recent (non-timestamped) files
- `'20260310_111531'` — loads a specific timestamped run (trade_log, reject_log, runtime_log)
- `'trade_log_20260310_111531.csv'` — loads by full filename

Works for both **backtest** and **live** runs.

In [ ]:
# == Cell 1: Initialize & list all ObjectStore files ==
from AlgorithmImports import *
import pandas as pd
import numpy as np
import io

qb = QuantBook()

# List everything in ObjectStore
try:
    all_keys = sorted([str(k) for k in qb.object_store.get_file_paths()])
except:
    all_keys = sorted(list(qb.object_store.keys))

trade_logs = [k for k in all_keys if 'trade_log' in k and k.endswith('.csv')]
reject_logs = [k for k in all_keys if 'reject_log' in k and k.endswith('.csv')]
runtime_logs = [k for k in all_keys if 'runtime_log' in k and k.endswith('.txt')]
universe_logs = [k for k in all_keys if 'auto_universe' in k and k.endswith('.csv')]
other_files = [k for k in all_keys if k not in trade_logs + reject_logs + runtime_logs + universe_logs]

print('=== ObjectStore Contents ===\n')
print(f'Trade Logs ({len(trade_logs)}):')
for k in trade_logs:
    print(f'  {k}')
print(f'\nReject Logs ({len(reject_logs)}):')
for k in reject_logs:
    print(f'  {k}')
print(f'\nRuntime Logs ({len(runtime_logs)}):')
for k in runtime_logs:
    print(f'  {k}')
print(f'\nUniverse Logs ({len(universe_logs)}):')
for k in universe_logs:
    print(f'  {k}')
if other_files:
    print(f'\nOther ({len(other_files)}):')
    for k in other_files:
        print(f'  {k}')

print('\n--- Set RUN_KEY in the next cell to load a specific run ---')

In [ ]:
# == Cell 2: Load trade log for a specific run ==
# Set to 'latest' for most recent, or a timestamp like '20260310_111531'
# or a full key like 'trade_log_20260310_111531.csv'
RUN_KEY = 'latest'

# Derive all three keys from RUN_KEY
if RUN_KEY == 'latest':
    trade_key = 'trade_log.csv'
    reject_key = 'reject_log.csv'
    runtime_key = 'runtime_log.txt'
elif RUN_KEY.startswith('trade_log_'):
    trade_key = RUN_KEY
    ts = RUN_KEY.replace('trade_log_', '').replace('.csv', '')
    reject_key = f'reject_log_{ts}.csv'
    runtime_key = f'runtime_log_{ts}.txt'
else:
    # Assume bare timestamp like '20260310_111531'
    ts = RUN_KEY
    trade_key = f'trade_log_{ts}.csv'
    reject_key = f'reject_log_{ts}.csv'
    runtime_key = f'runtime_log_{ts}.txt'

print(f'Looking for:')
print(f'  Trade:   {trade_key}')
print(f'  Reject:  {reject_key}')
print(f'  Runtime: {runtime_key}')
print()

# Load trade log
df = None
if qb.object_store.contains_key(trade_key):
    csv_data = qb.object_store.read(trade_key)
    df = pd.read_csv(io.StringIO(csv_data))
    print(f'Loaded {len(df)} trades from: {trade_key}')
else:
    print(f'Trade log not found: {trade_key}')

# Load reject log
rdf = None
if qb.object_store.contains_key(reject_key):
    csv_data = qb.object_store.read(reject_key)
    rdf = pd.read_csv(io.StringIO(csv_data))
    print(f'Loaded {len(rdf)} rejections from: {reject_key}')
else:
    print(f'Reject log not found: {reject_key}')

# Load runtime log
runtime_text = None
if qb.object_store.contains_key(runtime_key):
    runtime_text = qb.object_store.read(runtime_key)
    line_count = len(runtime_text.strip().split('\n')) if runtime_text.strip() else 0
    print(f'Loaded runtime log: {runtime_key} ({line_count} lines)')
else:
    print(f'Runtime log not found: {runtime_key}')

In [ ]:
# == Cell 2b: Runtime Log Viewer ==
if runtime_text:
    lines = runtime_text.strip().split('\n')
    print(f'=== RUNTIME LOG ({len(lines)} lines) ===\n')

    # Show key events (skip repetitive catchup retries)
    seen_catchup = 0
    for line in lines:
        # Summarize repeated catchup lines
        if '[CATCHUP] Running run_gap_scanner' in line or '[CATCHUP] Running load_universe_from_sheet' in line:
            seen_catchup += 1
            if seen_catchup <= 3:
                print(line)
            elif seen_catchup == 4:
                print(f'  ... (suppressing further catchup retries)')
            continue
        if seen_catchup > 3 and '[CATCHUP]' in line and 'Running' in line:
            continue
        seen_catchup = 0
        print(line)
else:
    print('No runtime log loaded.')

In [ ]:
# == Cell 3: Performance Dashboard ==
if df is not None and len(df) > 0:
    winners = df[df['pnl'] > 0]
    losers = df[df['pnl'] <= 0]
    total_pnl = df['pnl'].sum()
    gross_profit = winners['pnl'].sum() if len(winners) > 0 else 0
    gross_loss = abs(losers['pnl'].sum()) if len(losers) > 0 else 0
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

    print('=== PERFORMANCE DASHBOARD ===\n')
    print(f'Total Trades:     {len(df)}')
    print(f'Winners:          {len(winners)} ({len(winners)/len(df)*100:.1f}%)')
    print(f'Losers:           {len(losers)} ({len(losers)/len(df)*100:.1f}%)')
    print(f'Net P&L:          ${total_pnl:,.2f}')
    print(f'Profit Factor:    {profit_factor:.2f}')
    if len(winners) > 0:
        print(f'Avg Winner:       ${winners["pnl"].mean():,.2f}')
        print(f'Largest Win:      ${winners["pnl"].max():,.2f}')
    if len(losers) > 0:
        print(f'Avg Loser:        ${losers["pnl"].mean():,.2f}')
        print(f'Largest Loss:     ${losers["pnl"].min():,.2f}')
    print(f'Avg Duration:     {df["duration_bars"].mean():.1f} bars')

    # Direction breakdown
    print('\n--- By Direction ---')
    for d in ['LONG', 'SHORT']:
        sub = df[df['direction'] == d]
        if len(sub) == 0:
            continue
        w = sub[sub['pnl'] > 0]
        wr = len(w) / len(sub) * 100
        print(f'  {d:5s}: {len(sub):3d} trades | P&L ${sub["pnl"].sum():>8,.2f} | WR {wr:.1f}% | Avg ${sub["pnl"].mean():>7,.2f}')

    # Daily P&L
    print('\n--- Daily P&L ---')
    daily = df.groupby('date').agg(
        trades=('pnl', 'count'),
        pnl=('pnl', 'sum'),
        wins=('pnl', lambda x: (x > 0).sum()),
    ).round(2)
    daily['cumulative'] = daily['pnl'].cumsum()
    display(daily)
else:
    print('No trade data loaded.')

In [ ]:
# == Cell 4: Exit Reason Analysis ==
if df is not None and len(df) > 0:
    print('=== EXIT REASON BREAKDOWN ===\n')
    exit_summary = df.groupby('reason').agg(
        count=('pnl', 'count'),
        total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean() * 100),
        avg_duration=('duration_bars', 'mean'),
    ).round(2)
    exit_summary = exit_summary.sort_values('count', ascending=False)
    display(exit_summary)

    # Exit reason by direction
    print('\n--- Exit Reason x Direction ---')
    cross = df.groupby(['direction', 'reason']).agg(
        count=('pnl', 'count'),
        total_pnl=('pnl', 'sum'),
        win_rate=('pnl', lambda x: (x > 0).mean() * 100),
    ).round(2)
    display(cross)

In [ ]:
# == Cell 5: MAE / MFE / Capture Analysis ==
if df is not None and len(df) > 0:
    print('=== MAE / MFE ANALYSIS ===\n')

    for label, sub in [('Winners', df[df['pnl'] > 0]), ('Losers', df[df['pnl'] <= 0])]:
        if len(sub) == 0:
            continue
        print(f'--- {label} ({len(sub)} trades) ---')
        print(f'  Avg MAE:          {sub["mae_pct"].mean():.3f}%')
        print(f'  Avg MFE:          {sub["mfe_pct"].mean():.3f}%')
        print(f'  Avg MFE Captured: {sub["mfe_captured_pct"].mean():.1f}%')
        print(f'  Avg Duration:     {sub["duration_bars"].mean():.1f} bars')
        print()

    print('--- Top 5 Winners ---')
    top5w = df.nlargest(5, 'pnl')[['date', 'symbol', 'direction', 'pnl', 'pnl_pct', 'mfe_captured_pct', 'reason', 'duration_bars']]
    display(top5w)

    print('\n--- Top 5 Losers ---')
    top5l = df.nsmallest(5, 'pnl')[['date', 'symbol', 'direction', 'pnl', 'pnl_pct', 'mae_pct', 'reason', 'duration_bars']]
    display(top5l)

In [ ]:
# == Cell 6: Symbol Performance ==
if df is not None and len(df) > 0:
    print('=== SYMBOL PERFORMANCE ===\n')
    sym = df.groupby('symbol').agg(
        trades=('pnl', 'count'),
        total_pnl=('pnl', 'sum'),
        avg_pnl=('pnl', 'mean'),
        win_rate=('pnl', lambda x: (x > 0).mean() * 100),
        avg_mfe_cap=('mfe_captured_pct', 'mean'),
    ).round(2)
    sym = sym.sort_values('total_pnl', ascending=False)
    display(sym)

In [ ]:
# == Cell 7: Counterfactual Take-Profit Analysis ==
if df is not None and len(df) > 0:
    print('=== COUNTERFACTUAL TAKE PROFIT ===\n')
    print('(Tracks whether TP levels WOULD have been hit, regardless of USE_TAKE_PROFIT setting)\n')

    tp_cols = {
        '0.5R': 'cf_r0p5_hit',
        '1.0R': 'cf_r1_hit',
        '1.5R': 'cf_r1p5_hit',
        '2.0R': 'cf_r2_hit',
        '3.0R': 'cf_r3_hit',
    }

    rows = []
    for label, col in tp_cols.items():
        if col in df.columns:
            hits = df[col].sum()
            rate = hits / len(df) * 100
            hit_trades = df[df[col] == True]
            if len(hit_trades) > 0 and 'orb_range' in df.columns:
                r_mult = float(label.replace('R', ''))
                theoretical = (hit_trades['orb_range'] * r_mult * hit_trades['shares']).sum()
            else:
                theoretical = 0
            rows.append({'Target': label, 'Hits': int(hits), 'Rate': f'{rate:.1f}%', 'Theoretical $': f'${theoretical:,.2f}'})

    if rows:
        display(pd.DataFrame(rows).set_index('Target'))

    if 'trail_activated' in df.columns:
        act = df['trail_activated'].sum()
        print(f'\nTrail Activation Rate: {act}/{len(df)} ({act/len(df)*100:.1f}%)')

    tp_actual_cols = ['tp1_hit', 'tp2_hit', 'tp3_hit']
    if all(c in df.columns for c in tp_actual_cols):
        tp1 = df['tp1_hit'].sum() if df['tp1_hit'].dtype != object else (df['tp1_hit'] == True).sum()
        tp2 = df['tp2_hit'].sum() if df['tp2_hit'].dtype != object else (df['tp2_hit'] == True).sum()
        tp3 = df['tp3_hit'].sum() if df['tp3_hit'].dtype != object else (df['tp3_hit'] == True).sum()
        if tp1 + tp2 + tp3 > 0:
            print(f'\nActual TP Fills: TP1={int(tp1)} TP2={int(tp2)} TP3={int(tp3)}')

In [ ]:
# == Cell 8: Entry Filter Attribution ==
if df is not None and len(df) > 0:
    print('=== ENTRY FILTER ATTRIBUTION ===\n')
    print('Shows counterfactual: what if each filter was toggled differently?\n')

    cf_filters = {
        'EMA Align': 'cf_ema_align_pass',
        'VWAP': 'cf_vwap_pass',
        'Higher Close': 'cf_higher_close_pass',
        'Higher Open': 'cf_higher_open_pass',
        'Volume Rising': 'cf_volume_rising_pass',
        'Max Wick': 'cf_max_wick_pass',
        'Entry Window': 'cf_entry_window_pass',
        'Gap Direction': 'cf_gap_direction_pass',
    }

    for direction in ['LONG', 'SHORT']:
        sub = df[df['direction'] == direction]
        if len(sub) == 0:
            continue
        print(f'--- {direction} ({len(sub)} trades) ---')
        rows = []
        for name, col in cf_filters.items():
            if col not in sub.columns:
                continue
            passing = sub[sub[col] == True]
            failing = sub[sub[col] == False]
            rows.append({
                'Filter': name,
                'Pass': len(passing),
                'Fail': len(failing),
                'Pass P&L': f'${passing["pnl"].sum():,.2f}' if len(passing) > 0 else '$0',
                'Fail P&L': f'${failing["pnl"].sum():,.2f}' if len(failing) > 0 else '$0',
                'Pass WR': f'{(passing["pnl"] > 0).mean()*100:.0f}%' if len(passing) > 0 else '-',
                'Fail WR': f'{(failing["pnl"] > 0).mean()*100:.0f}%' if len(failing) > 0 else '-',
            })
        if rows:
            display(pd.DataFrame(rows).set_index('Filter'))
        print()

In [ ]:
# == Cell 9: Signal Rejection Analysis (from ObjectStore) ==
if rdf is not None and len(rdf) > 0:
    print('=== SIGNAL REJECTION ANALYSIS ===\n')
    print(f'Total unique rejections (first per symbol/reason/day): {len(rdf)}\n')

    print('--- Rejections by Reason ---')
    reason_counts = rdf.groupby('reason').agg(
        count=('symbol', 'count'),
        unique_symbols=('symbol', 'nunique'),
    ).sort_values('count', ascending=False)
    display(reason_counts)

    print('\n--- Most Rejected Symbols ---')
    sym_rej = rdf.groupby('symbol').agg(
        rejections=('reason', 'count'),
        reasons=('reason', lambda x: ', '.join(sorted(set(x)))),
    ).sort_values('rejections', ascending=False).head(15)
    display(sym_rej)

    print('\n--- Reason x Direction ---')
    cross = rdf.groupby(['direction', 'reason']).size().unstack(fill_value=0)
    display(cross)
else:
    print('No rejection log loaded. Run a backtest with the updated code first.')

In [ ]:
# == Cell 12: Download All Files ==
from IPython.display import HTML
import base64

def make_download_link(content, filename, label=None):
    b64 = base64.b64encode(content.encode()).decode()
    mime = 'text/csv' if filename.endswith('.csv') else 'text/plain'
    label = label or filename
    return f'<a download="{filename}" href="data:{mime};base64,{b64}" target="_blank">Download {label}</a>'

links = []
if df is not None and len(df) > 0:
    links.append(make_download_link(df.to_csv(index=False), 'trade_log.csv', f'trade_log.csv ({len(df)} trades)'))
if rdf is not None and len(rdf) > 0:
    links.append(make_download_link(rdf.to_csv(index=False), 'reject_log.csv', f'reject_log.csv ({len(rdf)} rejections)'))
if runtime_text:
    line_count = len(runtime_text.strip().split('\n'))
    links.append(make_download_link(runtime_text, 'runtime_log.txt', f'runtime_log.txt ({line_count} lines)'))

if links:
    display(HTML('<br>'.join(links)))
else:
    print('No data to download.')

In [ ]:
# == Cell 11: Download CSVs ==
from IPython.display import HTML
import base64

def make_download_link(dataframe, filename):
    csv_str = dataframe.to_csv(index=False)
    b64 = base64.b64encode(csv_str.encode()).decode()
    return f'<a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank">Download {filename} ({len(dataframe)} rows)</a>'

links = []
if df is not None and len(df) > 0:
    links.append(make_download_link(df, 'trade_log.csv'))
if rdf is not None and len(rdf) > 0:
    links.append(make_download_link(rdf, 'reject_log.csv'))

if links:
    display(HTML('<br>'.join(links)))
else:
    print('No data to download.')